# Proyecto 2 Bioseñales 

**Estudiantes:**

Luisa María Hernández Quintero 

Karen Agudelo Toro 

In [1]:
import mne
import matplotlib.pyplot as plt
import numpy as np
from tabulate import tabulate
import pandas as pd
import os
import mne
import os
import glob
import numpy as np
import pandas as pd
from tabulate import tabulate

In [9]:
import os
import mne

def procesar_archivo(ruta):
    nombre = os.path.basename(ruta)
    partes = nombre.split('_')
    
    # Extraemos el sujeto (ej. "S001") y el run eliminando la extensión (ej. "R04")
    sujeto_puro = partes[0]
    run_con_extension = partes[2].split('-')[1]
    run_puro = run_con_extension.split('.')[0]
    
    # Unimos el sujeto junto con el run (ej. "S001R04")
    sujeto_run = f"{sujeto_puro}{run_puro}"
    
    # Como todos tus archivos son estrictamente de este tipo, la condición siempre es "imaginacion"
    condicion = "imaginacion"
    
    # Carga de datos
    raw = mne.io.read_raw_eeglab(ruta, preload=True, verbose=False)
    
    # Selección de canales C3, Cz y C4
    raw.pick(['C3', 'Cz', 'C4'])
    
    # Filtrado (Notch a 60Hz y pasa-banda de 0.5 a 45 Hz)
    raw.notch_filter(60, verbose=False)
    raw.filter(0.5, 45, verbose=False)
    
    # Manejo de eventos
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    
    # Extracción de tareas de la misma manera
    eventos_dict = {
        'reposo':    event_id.get('TASK1T0') or event_id.get('TASK2T0'),
        'izquierda': event_id.get('TASK1T1') or event_id.get('TASK2T1'),
        'derecha':   event_id.get('TASK1T2') or event_id.get('TASK2T2')
    }
    
    # Limpiar eventos que no existan en este archivo específico
    eventos_dict = {k: v for k, v in eventos_dict.items() if v is not None}
    
    # Validación de seguridad por si acaso
    if not eventos_dict:
        print(f"Advertencia: No se encontraron eventos esperados en {nombre}")
        return None, sujeto_run, run_puro, condicion

    # Creación de épocas (0 a 2 segundos)
    epocas = mne.Epochs(
        raw, events, event_id=eventos_dict,
        tmin=0, tmax=2,
        baseline=None,
        preload=True,
        verbose=False
    )
    
    # Retorna los elementos solicitados manteniendo el orden
    return epocas, sujeto_run, run_puro, condicion

In [5]:
#Calcular PSD para cada época
def calcular_psd_epochs(epocas):
    psds = epocas.compute_psd(
        method='welch',
        fmin=0,
        fmax=45,
        n_fft=256,
        n_overlap=128,
        verbose=False
    )
    
    return psds.get_data(), psds.freqs

#funcion pot
def potencia_banda(psds, freqs, fmin, fmax):
    idx = (freqs >= fmin) & (freqs <= fmax)
    # promedio sobre la dimensión de frecuencia
    return np.mean(psds[:, :, idx], axis=2)

In [6]:
def construir_dataframe(epocas, sujeto, run, condicion):
        
    filas = []
    canales = epocas.ch_names
    
    for tarea in ['reposo', 'izquierda', 'derecha']:
        
        if tarea not in epocas.event_id:
            continue
        
        ep = epocas[tarea]
        
        
        psds, freqs = calcular_psd_epochs(ep)
        
        mu = potencia_banda(psds, freqs, 8, 13)
        beta = potencia_banda(psds, freqs, 13, 30)
        
        n_epochs = mu.shape[0]
        
        for i in range(n_epochs):
            for ch_idx, canal in enumerate(canales):
                
                filas.append({
                    'sujeto': sujeto,
                    'run': run,
                    'condicion': condicion, 
                    'tarea': tarea,
                    'canal': canal,
                    'mu': mu[i, ch_idx],
                    'beta': beta[i, ch_idx]
                })
    
    return pd.DataFrame(filas)

In [7]:
import glob

# 1. Definir la ruta de la carpeta
carpeta_sujetos = 'sujetos'

# 2. Obtener la lista de todos los archivos .set en esa carpeta
# Usamos glob para listar archivos que terminen en .set
archivos = glob.glob(os.path.join(carpeta_sujetos, "*.set"))

# Lista para ir guardando los DataFrames de cada archivo
lista_dfs = []

print(f"Se encontraron {len(archivos)} archivos. Iniciando procesamiento...")

# 3. Bucle para procesar cada archivo
for ruta in archivos:
    try:
        # Llamamos a tu primera función
        epocas, sujeto, run, condicion = procesar_archivo(ruta)
        
        # Llamamos a tu segunda función para obtener el DF de este archivo
        df_archivo = construir_dataframe(epocas, sujeto, run, condicion)
        
        # Guardamos el resultado en la lista
        lista_dfs.append(df_archivo)
        
        # print(f"Procesado con éxito: {os.path.basename(ruta)}")
        
    except Exception as e:
        print(f"Error procesando {ruta}: {e}")

# 4. Concatenar todos los DataFrames en uno solo
if lista_dfs:
    df_final = pd.concat(lista_dfs, ignore_index=True)
    print("\n¡Procesamiento completado!")
    print(f"Tamaño total del DataFrame: {df_final.shape}")
    
    # Opcional: Guardar a CSV
    # df_final.to_csv("resultados_eeg_motor.csv", index=False)
else:
    print("No se procesó ningún archivo.")

# Ver los primeros datos
# print(df_final.head())

df_mostrar = df_final.head(10).round(4) # Redondea a 4 decimales
from tabulate import tabulate



if lista_dfs:
    df_final = pd.concat(lista_dfs, ignore_index=True)
    
    print("\n" + "="*50)
    print("RESUMEN DEL DATAFRAME FINAL")
    print("="*50)
    
    
    print(tabulate(df_final.head(100), headers='keys', tablefmt='grid', showindex=False))
    
    print(f"\n... total de filas: {len(df_final)}")
else:
    print("No hay datos para mostrar.")
    

Se encontraron 30 archivos. Iniciando procesamiento...

¡Procesamiento completado!
Tamaño total del DataFrame: (2700, 7)

RESUMEN DEL DATAFRAME FINAL
+----------+-------+-------------+-----------+---------+-------------+-------------+
| sujeto   |   run | condicion   | tarea     | canal   |          mu |        beta |
+==========+=======+=============+===========+=========+=============+=============+
| sub-001  |    12 | imaginacion | reposo    | C3      | 1.61649e-11 | 8.18265e-12 |
+----------+-------+-------------+-----------+---------+-------------+-------------+
| sub-001  |    12 | imaginacion | reposo    | Cz      | 1.46045e-11 | 6.6918e-12  |
+----------+-------+-------------+-----------+---------+-------------+-------------+
| sub-001  |    12 | imaginacion | reposo    | C4      | 1.6771e-11  | 4.66084e-12 |
+----------+-------+-------------+-----------+---------+-------------+-------------+
| sub-001  |    12 | imaginacion | reposo    | C3      | 1.8205e-11  | 5.1088e-12  |
